## Topic Keyword Inspection and Naming

### Model Topic Keyword Analysis

In [ ]:
import pandas as pd
import os
from tabulate import tabulate

def get_enhanced_topics_for_dataset(dataset_name):
    """Get enhanced topics for a specified dataset

    Priority: Manual Naming > LLM Naming > No Naming

    Args:
        dataset_name: Dataset name (e.g., 'operation_positive')

    Returns:
        tuple: (enhanced_topics_dict, source_name)
    """
    # Check manual naming (highest priority)
    manual_var = f'{dataset_name}_enhanced'
    if manual_var in globals() and globals()[manual_var]:
        enhanced_topics = globals()[manual_var]
        return enhanced_topics, "人工命名"

    # Check LLM naming
    llm_var = f'{dataset_name}_enhanced_llm'
    if llm_var in globals() and globals()[llm_var]:
        enhanced_topics = globals()[llm_var]
        return enhanced_topics, "LLM命名"

    # Check other possible naming variables (backward compatibility)
    alt_vars = [
        f'{dataset_name}_topics_enhanced',
        f'{dataset_name}_manual_enhanced'
    ]

    for alt_var in alt_vars:
        if alt_var in globals() and globals()[alt_var]:
            enhanced_topics = globals()[alt_var]
            return enhanced_topics, "備用命名"

    return None, "無命名"

def analyze_model_topics(model, model_name, dataset_name=None, enhanced_topics=None, top_n=15):
    """Analyze topic keywords for a single model

    Args:
        model: BERTopic model
        model_name: Model display name
        dataset_name: Dataset name, used to automatically retrieve enhanced topics
        enhanced_topics: Directly provided enhanced topics dictionary (if any)
        top_n: Output top N topics

    Returns:
        Topic analysis result dictionary
    """
    print(f"\n{'='*60}")
    print(f"分析 {model_name} 模型")
    print(f"{'='*60}")

    # Get topic information
    topic_info = model.get_topic_info()
    non_outlier_topics = topic_info[topic_info['Topic'] != -1].sort_values("Count", ascending=False)

    # Basic statistics
    total_docs = topic_info['Count'].sum()
    outlier_count = topic_info[topic_info['Topic'] == -1]['Count'].values[0] if -1 in topic_info['Topic'].values else 0
    valid_topics = len(non_outlier_topics)
    coverage_rate = (total_docs - outlier_count) / total_docs * 100

    print(f"模型基本統計:")
    print(f"  總文檔數: {total_docs}")
    print(f"  有效主題數: {valid_topics}")
    print(f"  離群文檔: {outlier_count} ({outlier_count/total_docs*100:.1f}%)")
    print(f"  覆蓋率: {coverage_rate:.1f}%")

    # Get enhanced topics
    naming_source = "無命名"
    if enhanced_topics is None and dataset_name:
        enhanced_topics, naming_source = get_enhanced_topics_for_dataset(dataset_name)
    elif enhanced_topics is not None:
        naming_source = "直接提供"

    # Check status of enhanced topics
    if enhanced_topics:
        print(f"  增強主題: 已載入 {len(enhanced_topics)} 個命名 ({naming_source})")
    else:
        print(f"  增強主題: 未載入，將使用關鍵詞組合命名")

    # Take top N topics
    top_topics = non_outlier_topics.head(top_n)

    print(f"\n前 {len(top_topics)} 個最大主題詳細分析:")
    print("-" * 80)

    # Prepare CSV data
    csv_data = []

    for idx, (_, row) in enumerate(top_topics.iterrows(), 1):
        topic_id = row['Topic']
        topic_size = row['Count']
        topic_percentage = topic_size / total_docs * 100

        # Get topic keywords (top 10)
        topic_keywords = model.get_topic(topic_id)[:10]

        # Topic Name
        if enhanced_topics and topic_id in enhanced_topics:
            topic_name = enhanced_topics[topic_id]['name']
            name_source = naming_source
        else:
            # Use top 2 keywords as topic name
            top_words = [word for word, _ in topic_keywords[:2]]
            topic_name = "-".join(top_words)
            name_source = "關鍵詞組合"

        print(f"\n【主題 {idx}】主題ID: {topic_id}")
        print(f"主題名稱: {topic_name} ({name_source})")
        print(f"文檔數量: {topic_size} ({topic_percentage:.1f}%)")
        print(f"關鍵詞:")

        # Display keywords
        for i, (word, weight) in enumerate(topic_keywords, 1):
            print(f"  {i:2d}. {word} ({weight:.4f})")

        # Prepare CSV row data
        csv_row = {
            '排名': idx,
            '主題ID': topic_id,
            '主題名稱': topic_name,
            '名稱來源': name_source,
            '文檔數': topic_size,
            '佔比(%)': f"{topic_percentage:.2f}%",
            '模型名稱': model_name,
            '資料集': dataset_name if dataset_name else "未指定"
        }

        # Add keyword columns
        for i, (word, weight) in enumerate(topic_keywords, 1):
            csv_row[f'關鍵詞{i}'] = word
            csv_row[f'權重{i}'] = f"{weight:.4f}"

        csv_data.append(csv_row)

    return {
        'model_name': model_name,
        'dataset_name': dataset_name,
        'total_docs': total_docs,
        'valid_topics': valid_topics,
        'outlier_count': outlier_count,
        'coverage_rate': coverage_rate,
        'enhanced_count': len(enhanced_topics) if enhanced_topics else 0,
        'naming_source': naming_source,
        'csv_data': csv_data
    }

def verify_models_exist():
    """Verify if all expected models exist"""
    print("驗證模型存在性...")
    print("-" * 40)

    expected_models = [
        ('operation_positive_model', 'operation_positive', '營運活動正面情緒'),
        ('operation_negative_model', 'operation_negative', '營運活動負面情緒'),
        ('ip_positive_model', 'ip_positive', 'IP聯動正面情緒'),
        ('ip_negative_model', 'ip_negative', 'IP聯動負面情緒'),
        ('balance_update_model', 'balance_update', '平衡性版本更新'),
    ]

    available_models = []
    missing_models = []

    for model_var, dataset_name, display_name in expected_models:
        if model_var in globals() and globals()[model_var] is not None:
            available_models.append((model_var, dataset_name, display_name))
            print(f"  找到: {display_name}")
        else:
            missing_models.append((model_var, dataset_name, display_name))
            print(f"  缺失: {display_name}")

    print(f"\n可用模型: {len(available_models)}/{len(expected_models)}")

    if missing_models:
        print("缺失的模型:")
        for model_var, _, display_name in missing_models:
            print(f"  - {display_name} ({model_var})")
        print("\n請先執行模型載入步驟 (9.0) 或模型訓練步驟 (8.1-8.5)")

    return available_models

def analyze_all_models():
    """Analyze all trained models"""
    print("開始分析所有模型的主題關鍵詞...")
    print("=" * 80)

    # First verify model existence
    available_models = verify_models_exist()

    if not available_models:
        print("錯誤: 沒有找到任何可用的模型")
        return []

    all_results = []
    all_csv_data = []

    for model_var, dataset_name, display_name in available_models:
        print(f"\n{'='*50}")
        print(f"處理模型: {display_name}")
        print('='*50)

        try:
            model = globals()[model_var]
            print(f"模型已載入: {model_var}")

            # Analyze model - Pass dataset_name to automatically retrieve enhanced topics
            result = analyze_model_topics(
                model=model,
                model_name=display_name,
                dataset_name=dataset_name,
                top_n=15
            )
            all_results.append(result)

            # Merge CSV data
            all_csv_data.extend(result['csv_data'])

        except Exception as e:
            print(f"分析 {display_name} 時發生錯誤: {str(e)}")
            continue

    # Generate summary report
    if all_results:
        print(f"\n\n{'='*80}")
        print("所有模型總結報告")
        print("="*80)

        summary_data = []
        for result in all_results:
            summary_data.append([
                result['model_name'],
                result['total_docs'],
                result['valid_topics'],
                f"{result['outlier_count']} ({result['outlier_count']/result['total_docs']*100:.1f}%)",
                f"{result['coverage_rate']:.1f}%",
                f"{result['enhanced_count']}個",
                result['naming_source']
            ])

        headers = ['模型名稱', '總文檔數', '有效主題數', '離群文檔', '覆蓋率', '已命名主題', '命名來源']
        print(tabulate(summary_data, headers=headers, tablefmt="grid"))

        # Save detailed CSV report
        if all_csv_data:
            save_analysis_results(all_results, all_csv_data)

    return all_results

def save_analysis_results(all_results, all_csv_data):
    """Save analysis results to CSV file"""
    try:
        # Check if PIPELINE_CONFIG exists
        if 'PIPELINE_CONFIG' not in globals():
            print("警告: PIPELINE_CONFIG 未定義，無法儲存結果")
            return

        results_dir = os.path.join(PIPELINE_CONFIG['data']['base_path'], PIPELINE_CONFIG['output']['results_dir'])
        os.makedirs(results_dir, exist_ok=True)

        # Save merged analysis results
        csv_path = os.path.join(results_dir, 'all_models_topics_keywords_analysis.csv')
        df = pd.DataFrame(all_csv_data)
        df.to_csv(csv_path, index=False, encoding='utf-8-sig')
        print(f"\n詳細主題關鍵詞分析已儲存至: {csv_path}")

        # Save individual CSV for each model
        for result in all_results:
            if result['csv_data']:
                model_csv_data = result['csv_data']
                safe_model_name = result['model_name'].replace('/', '_').replace(' ', '_').replace('-', '_')
                model_csv_path = os.path.join(results_dir, f"{safe_model_name}_topics_keywords.csv")
                model_df = pd.DataFrame(model_csv_data)
                model_df.to_csv(model_csv_path, index=False, encoding='utf-8-sig')
                print(f"{result['model_name']} 主題分析已儲存至: {os.path.basename(model_csv_path)}")

    except Exception as e:
        print(f"儲存分析結果時發生錯誤: {str(e)}")

def generate_keywords_summary():
    """Generate keyword statistics summary"""
    print(f"\n{'='*80}")
    print("生成跨模型關鍵詞統計摘要...")
    print("="*80)

    # Verify model existence
    available_models = verify_models_exist()
    if not available_models:
        print("錯誤: 沒有找到任何可用的模型")
        return {}

    keyword_frequency = {}

    for model_var, dataset_name, display_name in available_models:
        try:
            model = globals()[model_var]
            topic_info = model.get_topic_info()

            # Get all keywords from top 10 topics
            top_topics = topic_info[topic_info['Topic'] != -1].sort_values("Count", ascending=False)['Topic'][:10]

            print(f"\n{display_name} 前10個主題的高頻關鍵詞:")
            model_keywords = []

            for topic_id in top_topics:
                keywords = model.get_topic(topic_id)[:5]  # Take top 5 keywords for each topic
                for word, weight in keywords:
                    model_keywords.append(word)
                    if word not in keyword_frequency:
                        keyword_frequency[word] = []
                    keyword_frequency[word].append((display_name, weight))

            # Display unique keywords for this model
            from collections import Counter
            word_counts = Counter(model_keywords)
            print("  高頻詞:", ', '.join([f"{word}({count})" for word, count in word_counts.most_common(15)]))

        except Exception as e:
            print(f"分析 {display_name} 關鍵詞時發生錯誤: {str(e)}")
            continue

    # Find common keywords across models
    print(f"\n跨模型共同關鍵詞分析:")
    cross_model_keywords = {word: models for word, models in keyword_frequency.items() if len(models) >= 2}

    if cross_model_keywords:
        print("出現在多個模型中的關鍵詞:")
        for word, models in sorted(cross_model_keywords.items(), key=lambda x: len(x[1]), reverse=True)[:20]:
            model_list = [model_name for model_name, _ in models]
            print(f"  {word}: {', '.join(model_list)} (出現在{len(models)}個模型)")
    else:
        print("未找到跨模型的共同關鍵詞")

    return keyword_frequency

def debug_enhanced_topics():
    """Debug function: Check all enhanced topic variables"""
    print("調試：檢查所有增強主題變數")
    print("="*50)

    datasets = ['operation_positive', 'operation_negative', 'ip_positive', 'ip_negative', 'balance_update']

    for dataset in datasets:
        print(f"\n資料集: {dataset}")

        # Check various possible variable names
        possible_vars = [
            f'{dataset}_enhanced',
            f'{dataset}_enhanced_llm',
            f'{dataset}_topics_enhanced',
            f'{dataset}_manual_enhanced'
        ]

        found_any = False
        for var_name in possible_vars:
            if var_name in globals():
                var_value = globals()[var_name]
                if var_value:
                    print(f"  找到: {var_name} = {len(var_value)} 個主題")
                    # Show top 3 topic names as samples
                    sample_topics = list(var_value.items())[:3]
                    for topic_id, topic_data in sample_topics:
                        topic_name = topic_data.get('name', '無名稱')
                        print(f"    主題{topic_id}: {topic_name}")
                    found_any = True
                else:
                    print(f"  空值: {var_name} = None 或空")

        if not found_any:
            print(f"  未找到任何增強主題變數")

def run_complete_analysis():
    """Execute complete topic keyword analysis pipeline"""
    print("="*80)
    print("開始進行 9.1 模型主題關鍵詞輸出分析")
    print("="*80)

    # First debug check variable status
    debug_enhanced_topics()

    # Analyze all models
    analysis_results = analyze_all_models()

    if analysis_results:
        # Generate keyword statistics summary
        keyword_stats = generate_keywords_summary()

        print(f"\n{'='*80}")
        print("9.1 模型主題關鍵詞分析完成")
        print("="*80)
        print(f"已分析 {len(analysis_results)} 個模型")
        print(f"詳細結果已儲存至 results 目錄")
        print(f"包含主題關鍵詞、權重、文檔數量等完整資訊")

        return analysis_results, keyword_stats
    else:
        print(f"\n{'='*80}")
        print("9.1 模型主題關鍵詞分析失敗")
        print("="*80)
        print("請確保：")
        print("1. 已執行模型載入步驟 (9.0)")
        print("2. 或已執行模型訓練步驟 (8.1-8.5)")
        print("3. 全域變數 PIPELINE_CONFIG 已定義")

        return [], {}

# Execute complete analysis
if __name__ == "__main__":
    analysis_results, keyword_stats = run_complete_analysis()

開始進行 9.1 模型主題關鍵詞輸出分析
調試：檢查所有增強主題變數

資料集: operation_positive
  找到: operation_positive_enhanced = 11 個主題
    主題0: 悟空神操作讚美
    主題1: 官方活動讚美
    主題2: 可愛造型期待

資料集: operation_negative
  找到: operation_negative_enhanced = 14 個主題
    主題0: 造型外觀不滿
    主題1: 遊戲公司商業抱怨
    主題2: 語音質量不滿

資料集: ip_positive
  找到: ip_positive_enhanced = 24 個主題
    主題0: 皇帝得獎祝賀
    主題1: 主播戰神讚美
    主題2: 主播黃金戰神讚美

資料集: ip_negative
  找到: ip_negative_enhanced = 19 個主題
    主題0: 虎杖抽獎結果討論
    主題1: 購買經濟負擔
    主題2: 虎杖抽獎失望

資料集: balance_update
  找到: balance_update_enhanced = 34 個主題
    主題0: 黑貓無敵主播討論
    主題1: 黃金戰神外觀討論
    主題2: 技能削弱討論
開始分析所有模型的主題關鍵詞...
驗證模型存在性...
----------------------------------------
  找到: 營運活動正面情緒
  找到: 營運活動負面情緒
  找到: IP聯動正面情緒
  找到: IP聯動負面情緒
  找到: 平衡性版本更新

可用模型: 5/5

處理模型: 營運活動正面情緒
模型已載入: operation_positive_model

分析 營運活動正面情緒 模型
模型基本統計:
  總文檔數: 918
  有效主題數: 11
  離群文檔: 323 (35.2%)
  覆蓋率: 64.8%
  增強主題: 已載入 11 個命名 (人工命名)

前 11 個最大主題詳細分析:
--------------------------------------------------------------------------------

【

### LLM Topic Naming

In [ ]:
import openai
from openai import OpenAI
import time

def check_llm_prerequisites():
    """Check prerequisites for LLM naming"""
    print("檢查LLM命名前置條件...")
    print("=" * 40)

    prerequisites_status = []
    all_ok = True

    # Check PIPELINE_CONFIG
    if 'PIPELINE_CONFIG' in globals():
        prerequisites_status.append(("PIPELINE_CONFIG", "存在"))
    else:
        prerequisites_status.append(("PIPELINE_CONFIG", "缺失"))
        all_ok = False

    # Check API Key
    if 'PIPELINE_CONFIG' in globals():
        api_key = PIPELINE_CONFIG.get('llm', {}).get('api_key')
        if api_key:
            api_key_display = f"已設定 ({api_key[:8]}...)"
            prerequisites_status.append(("OpenAI API Key", api_key_display))
        else:
            prerequisites_status.append(("OpenAI API Key", "未設定"))
            all_ok = False

    # Check enhancer
    if 'enhancer' in globals():
        enhancer = globals()['enhancer']
        if hasattr(enhancer, 'client') and enhancer.client is not None:
            prerequisites_status.append(("LLM Enhancer", "正常"))
        else:
            prerequisites_status.append(("LLM Enhancer", "客戶端未初始化"))
            all_ok = False
    else:
        prerequisites_status.append(("LLM Enhancer", "不存在"))
        all_ok = False

    # Check models
    expected_models = ['operation_positive_model', 'operation_negative_model',
                      'ip_positive_model', 'ip_negative_model', 'balance_update_model']
    available_models = sum(1 for model_var in expected_models if model_var in globals())
    prerequisites_status.append(("可用模型", f"{available_models}/{len(expected_models)}個"))

    if available_models == 0:
        all_ok = False

    # Print status
    for item, status in prerequisites_status:
        print(f"  {item}: {status}")

    return all_ok

def load_saved_models_for_llm():
    """Load saved models for LLM naming"""
    print("\n載入已儲存的模型...")
    print("=" * 40)

    if 'PIPELINE_CONFIG' not in globals():
        print("錯誤: PIPELINE_CONFIG 未定義")
        return {}

    models_dir = os.path.join(PIPELINE_CONFIG['data']['base_path'], PIPELINE_CONFIG['output']['models_dir'])

    # Model file mapping
    model_files = {
        'operation_positive': 'operation_positive_model.pkl',
        'operation_negative': 'operation_negative_model.pkl',
        'ip_positive': 'ip_positive_model.pkl',
        'ip_negative': 'ip_negative_model.pkl',
        'balance_update': 'balance_update_model.pkl'
    }

    loaded_models = {}

    for dataset_name, filename in model_files.items():
        model_path = os.path.join(models_dir, filename)
        model_var = f'{dataset_name}_model'

        # Check if file exists
        if os.path.exists(model_path):
            try:
                # Load model
                from bertopic import BERTopic
                model = BERTopic.load(model_path)

                # Ensure embedding model is set
                if not hasattr(model, 'embedding_model') or model.embedding_model is None:
                    if 'analyzer' in globals() and hasattr(globals()['analyzer'], 'embedding_model'):
                        model.embedding_model = globals()['analyzer'].embedding_model
                        print(f"  {dataset_name}: 已重新設定嵌入模型")

                # Save to global variables - Key fix point
                globals()[model_var] = model
                loaded_models[dataset_name] = model

                print(f"已載入 {dataset_name}: {filename}")

            except Exception as e:
                print(f"載入 {dataset_name} 失敗: {str(e)}")
        else:
            print(f"找不到 {dataset_name}: {filename}")

    print(f"\n成功載入 {len(loaded_models)} 個模型")
    return loaded_models

def enhance_single_model_topics(model, dataset_name, enhancer):
    """Name topics for a single model"""
    print(f"\n命名 {dataset_name} 主題...")
    print("-" * 30)

    try:
        # Get basic model info
        topic_info = model.get_topic_info()
        non_outlier_topics = topic_info[topic_info['Topic'] != -1]

        print(f"  模型有 {len(non_outlier_topics)} 個有效主題")

        # Perform topic naming
        enhanced = enhancer.enhance_model_topics(model, dataset_name)

        if enhanced:
            print(f"  成功命名 {len(enhanced)} 個主題")
            return enhanced
        else:
            print(f"  命名失敗，返回空結果")
            return {}

    except Exception as e:
        print(f"  命名過程中發生錯誤: {str(e)}")
        return {}

def enhance_all_topics():
    """Name topics for all trained models"""

    print("開始進行主題命名...")
    print("=" * 50)

    # Check prerequisites
    if not check_llm_prerequisites():
        print("\n錯誤: 前置條件檢查失敗")
        print("請確保：")
        print("1. PIPELINE_CONFIG 已定義")
        print("2. OpenAI API Key 已設定")
        print("3. enhancer 已初始化")
        print("4. 至少有一個模型可用")
        return None

    # Load models (if needed)
    print("\n檢查並載入模型...")
    model_vars = {
        'operation_positive': 'operation_positive_model',
        'operation_negative': 'operation_negative_model',
        'ip_positive': 'ip_positive_model',
        'ip_negative': 'ip_negative_model',
        'balance_update': 'balance_update_model'
    }

    # Check which models need to be loaded
    missing_models = []
    for dataset_name, model_var in model_vars.items():
        if model_var not in globals():
            missing_models.append(dataset_name)

    if missing_models:
        print(f"需要載入模型: {', '.join(missing_models)}")
        loaded_models = load_saved_models_for_llm()
    else:
        print("所有模型已在記憶體中")

    enhanced_results = {}
    success_count = 0
    total_count = 0

    # Get enhancer
    enhancer = globals()['enhancer']

    # Process each model
    for dataset_name, model_var in model_vars.items():
        total_count += 1

        if model_var in globals():
            model = globals()[model_var]

            # Perform topic naming
            enhanced = enhance_single_model_topics(model, dataset_name, enhancer)

            if enhanced:
                enhanced_results[dataset_name] = enhanced
                success_count += 1

                # Save enhanced results to global variables - Key fix point
                # Use consistent naming convention
                globals()[f'{dataset_name}_enhanced'] = enhanced  # Primary variable
                globals()[f'{dataset_name}_enhanced_llm'] = enhanced  # LLM-specific backup

                print(f"  命名結果已儲存至:")
                print(f"    - {dataset_name}_enhanced")
                print(f"    - {dataset_name}_enhanced_llm")
            else:
                print(f"  {dataset_name} 命名失敗")
        else:
            print(f"模型 {model_var} 未找到，跳過")

    # Summary report
    print(f"\n{'='*50}")
    print("LLM主題命名完成")

開始進行 9.2 LLM主題命名 - 完全修復版
開始進行主題命名...
檢查LLM命名前置條件...
  PIPELINE_CONFIG: 存在
  OpenAI API Key: 已設定 (sk-proj-...)
  LLM Enhancer: 正常
  可用模型: 5/5個

檢查並載入模型...
所有模型已在記憶體中

命名 operation_positive 主題...
------------------------------
  模型有 11 個有效主題
Enhancing topics for 營運活動正面情緒...
Topic 0: 悟空神操作讚美
Topic 1: 官方活動讚美
Topic 2: 可愛造型期待
Topic 3: 直購造型需求
Topic 4: 特效表現讚美
Topic 5: 免費活動期待
Topic 6: 遊戲趣味與可愛角色評價
Topic 7: 語音配音讚賞
Topic 8: 活動獎勵期待
Topic 9: 傳說對決週年期待
Topic 10: 牛牛復刻期待
  成功命名 11 個主題
  命名結果已儲存至:
    - operation_positive_enhanced
    - operation_positive_enhanced_llm

命名 operation_negative 主題...
------------------------------
  模型有 14 個有效主題
Enhancing topics for 營運活動負面情緒...
Topic 0: 造型外觀不滿
Topic 1: 遊戲公司商業抱怨
Topic 2: 語音質量不滿
Topic 3: 造型價格抱怨
Topic 4: 遊戲內溝通無力感
Topic 5: 配對機制改變不滿
Topic 6: 免費獲得需求
Topic 7: 造型特效不滿
Topic 8: 過去活動失望與未來擔憂
Topic 9: 寶箱與聖典造型不滿
Topic 10: 越南服獲取不滿
Topic 11: 盤子抽獎不滿
Topic 12: 抽獎價格不滿
Topic 13: 造型唯一不滿
  成功命名 14 個主題
  命名結果已儲存至:
    - operation_negative_enhanced
    - operation_negative_enhanced

### Manual Topic Naming

In [ ]:
import pandas as pd
import os
from tabulate import tabulate
import json

class ManualTopicNamer:
    """Manual Topic Namer"""

    def __init__(self, config):
        self.config = config
        self.manual_naming_history = {}
        self.results_dir = os.path.join(config['data']['base_path'], config['output']['results_dir'])
        os.makedirs(self.results_dir, exist_ok=True)

    def get_current_enhanced_topics(self, dataset_name):
        """Get current enhanced topics (priority: manual naming)"""
        # Priority: Manual Naming > LLM Naming > None
        manual_var = f'{dataset_name}_enhanced'
        llm_var = f'{dataset_name}_enhanced_llm'

        if manual_var in globals() and globals()[manual_var]:
            enhanced_topics = globals()[manual_var]
            source = "人工命名"
        elif llm_var in globals() and globals()[llm_var]:
            enhanced_topics = globals()[llm_var]
            source = "LLM命名"
        else:
            enhanced_topics = None
            source = "無命名"

        return enhanced_topics, source

    def display_current_topics(self, model, dataset_name, enhanced_topics=None, show_docs=True):
        """Display detailed information for 5 topics for manual review"""
        print(f"\n{'='*80}")
        print(f"主題詳細資訊 - {dataset_name} (每次顯示5個主題)")
        print(f"{'='*80}")

        topic_info = model.get_topic_info()
        non_outlier_topics = topic_info[topic_info['Topic'] != -1].sort_values("Count", ascending=False)

        # Display only the top 5 topics
        top_5_topics = non_outlier_topics.head(5)

        # If enhanced_topics is not provided, try to get it automatically
        if enhanced_topics is None:
            enhanced_topics, source = self.get_current_enhanced_topics(dataset_name)
            if enhanced_topics:
                print(f"已載入增強主題: {source} ({len(enhanced_topics)}個)")
            else:
                print(f"未找到增強主題")

        # Get all documents and topic assignments (if showing related documents is needed)
        if show_docs and hasattr(model, 'topics_'):
            # Load corresponding text data
            try:
                # Convert display name to dataset key
                dataset_key = self._convert_display_name_to_key(dataset_name)
                if 'processor' in globals():
                    texts = globals()['processor'].get_texts(dataset_key)
                    topics_list = model.topics_
                else:
                    texts = None
                    topics_list = None
            except Exception as e:
                print(f"載入文本資料失敗: {str(e)}")
                texts = None
                topics_list = None
        else:
            texts = None
            topics_list = None

        for idx, (_, row) in enumerate(top_5_topics.iterrows(), 1):
            topic_id = row['Topic']
            topic_size = row['Count']
            topic_percentage = topic_size / topic_info['Count'].sum() * 100

            print(f"\n【主題 {idx}】")
            print(f"主題ID: {topic_id}")
            print(f"文檔數: {topic_size} ({topic_percentage:.1f}%)")

            # Get current name
            if enhanced_topics and topic_id in enhanced_topics:
                current_name = enhanced_topics[topic_id]['name']
                name_source = "已命名"
            else:
                topic_keywords = model.get_topic(topic_id)[:2]
                current_name = "-".join([word for word, _ in topic_keywords])
                name_source = "待命名"

            print(f"當前名稱: {current_name} ({name_source})")

            # Get top 5 keywords
            topic_keywords = model.get_topic(topic_id)[:5]
            print(f"關鍵詞:")
            for i, (word, weight) in enumerate(topic_keywords, 1):
                print(f"  {i}. {word} (權重: {weight:.4f})")

            # Display 2 most relevant documents
            if texts and topics_list and len(texts) == len(topics_list):
                topic_docs = []
                for doc_idx, doc_topic in enumerate(topics_list):
                    if doc_topic == topic_id:
                        topic_docs.append((doc_idx, texts[doc_idx]))

                if len(topic_docs) >= 2:
                    print(f"相關文檔範例:")
                    for doc_num, (doc_idx, doc_text) in enumerate(topic_docs[:2], 1):
                        # Limit document length for display
                        display_text = doc_text[:100] + "..." if len(doc_text) > 100 else doc_text
                        print(f"  文檔{doc_num}: {display_text}")
                elif len(topic_docs) == 1:
                    display_text = topic_docs[0][1][:100] + "..." if len(topic_docs[0][1]) > 100 else topic_docs[0][1]
                    print(f"相關文檔範例:")
                    print(f"  文檔1: {display_text}")
                else:
                    print(f"相關文檔: 無法找到相關文檔")

            print("-" * 60)

        return top_5_topics

    def _convert_display_name_to_key(self, display_name):
        """Convert display name to dataset key"""
        mapping = {
            '營運活動正面情緒': 'operation_positive',
            '營運活動負面情緒': 'operation_negative',
            'IP聯動正面情緒': 'ip_positive',
            'IP聯動負面情緒': 'ip_negative',
            '平衡性版本更新': 'balance_update'
        }
        return mapping.get(display_name, display_name)

    def manual_rename_single(self, model, dataset_name, topic_id, new_name, enhanced_topics=None):
        """Manually rename a single topic"""
        topic_info = model.get_topic_info()

        if topic_id not in topic_info['Topic'].values:
            print(f"錯誤: 主題ID {topic_id} 不存在")
            return False

        # If enhanced_topics is not provided, get the current one
        if enhanced_topics is None:
            enhanced_topics, _ = self.get_current_enhanced_topics(dataset_name)
            if enhanced_topics is None:
                enhanced_topics = {}

        # Get original name
        if enhanced_topics and topic_id in enhanced_topics:
            old_name = enhanced_topics[topic_id]['name']
        else:
            topic_keywords = model.get_topic(topic_id)[:2]
            old_name = "-".join([word for word, _ in topic_keywords])

        # Preserve original keyword information
        if topic_id in enhanced_topics:
            enhanced_topics[topic_id]['name'] = new_name
        else:
            topic_keywords = model.get_topic(topic_id)[:8]
            enhanced_topics[topic_id] = {
                'name': new_name,
                'keywords': [word for word, _ in topic_keywords],
                'original_keywords': [f"{word}({weight:.3f})" for word, weight in topic_keywords[:5]]
            }

        # Record naming history
        if dataset_name not in self.manual_naming_history:
            self.manual_naming_history[dataset_name] = []

        self.manual_naming_history[dataset_name].append({
            'topic_id': topic_id,
            'old_name': old_name,
            'new_name': new_name,
            'timestamp': pd.Timestamp.now().isoformat()
        })

        print(f"完成: 主題 {topic_id} 已重命名")
        print(f"   舊名稱: {old_name}")
        print(f"   新名稱: {new_name}")

        return enhanced_topics

    def batch_rename_interface(self, model, dataset_name, enhanced_topics=None):
        """Batch rename interface - process 5 topics at a time"""
        print(f"\n批次重命名模式 - {dataset_name}")
        print("=" * 50)
        print("說明: 輸入格式為 '主題ID:新名稱'，每行一個")
        print("範例: 1:造型設計優化建議")
        print("輸入 'done' 完成保存，'cancel' 取消，'next' 查看下5個主題")
        print("-" * 50)

        # If enhanced_topics is not provided, get the current one
        if enhanced_topics is None:
            enhanced_topics, source = self.get_current_enhanced_topics(dataset_name)
            if enhanced_topics is None:
                enhanced_topics = {}
            else:
                print(f"已載入現有命名: {source} ({len(enhanced_topics)}個)")

        modified_count = 0
        start_index = 0

        # Get all topics
        topic_info = model.get_topic_info()
        all_topics = topic_info[topic_info['Topic'] != -1].sort_values("Count", ascending=False)

        while start_index < len(all_topics):
            # Display current 5 topics
            print(f"\n顯示第 {start_index+1}-{min(start_index+5, len(all_topics))} 個主題:")
            current_topics = all_topics.iloc[start_index:start_index+5]

            # Display topic details
            self._display_topic_batch(model, dataset_name, current_topics, enhanced_topics)

            print(f"\n請輸入要修改的主題 (格式: 主題ID:新名稱):")
            print("輸入 'done' 完成, 'cancel' 取消, 'next' 下5個主題, 'prev' 上5個主題")

            while True:
                try:
                    user_input = input(">>> ").strip()

                    if user_input.lower() == 'done':
                        if modified_count > 0:
                            self.save_manual_naming_results(dataset_name, enhanced_topics)
                        return enhanced_topics
                    elif user_input.lower() == 'cancel':
                        print("操作取消: 批次重命名已取消")
                        return enhanced_topics
                    elif user_input.lower() == 'next':
                        start_index += 5
                        break
                    elif user_input.lower() == 'prev':
                        start_index = max(0, start_index - 5)
                        break
                    elif ':' in user_input:
                        topic_id_str, new_name = user_input.split(':', 1)
                        topic_id = int(topic_id_str.strip())
                        new_name = new_name.strip()

                        if new_name:
                            enhanced_topics = self.manual_rename_single(
                                model, dataset_name, topic_id, new_name, enhanced_topics
                            )
                            if enhanced_topics:
                                modified_count += 1
                                print(f"已修改 {modified_count} 個主題")
                        else:
                            print("錯誤: 新名稱不能為空")
                    else:
                        print("錯誤: 格式錯誤，請使用 '主題ID:新名稱' 格式")

                except ValueError:
                    print("錯誤: 主題ID必須是數字")
                except KeyboardInterrupt:
                    print("\n操作已中斷")
                    return enhanced_topics
                except Exception as e:
                    print(f"錯誤: 發生錯誤: {str(e)}")

        print(f"\n完成: 已查看所有主題，共修改了 {modified_count} 個主題")
        if modified_count > 0:
            self.save_manual_naming_results(dataset_name, enhanced_topics)
        return enhanced_topics

    def _display_topic_batch(self, model, dataset_name, topics_batch, enhanced_topics):
        """Display details for a batch of topics"""
        # Load text data
        try:
            dataset_key = self._convert_display_name_to_key(dataset_name)
            if 'processor' in globals():
                texts = globals()['processor'].get_texts(dataset_key)
                topics_list = model.topics_
            else:
                texts = None
                topics_list = None
        except Exception as e:
            texts = None
            topics_list = None

        for idx, (_, row) in enumerate(topics_batch.iterrows(), 1):
            topic_id = row['Topic']
            topic_size = row['Count']

            print(f"\n【主題 {topic_id}】")
            print(f"文檔數: {topic_size}")

            # Current name
            if enhanced_topics and topic_id in enhanced_topics:
                current_name = enhanced_topics[topic_id]['name']
                print(f"當前名稱: {current_name} (已命名)")
            else:
                topic_keywords = model.get_topic(topic_id)[:2]
                current_name = "-".join([word for word, _ in topic_keywords])
                print(f"當前名稱: {current_name} (待命名)")

            # Top 5 keywords
            topic_keywords = model.get_topic(topic_id)[:5]
            keywords_str = ", ".join([f"{word}({weight:.3f})" for word, weight in topic_keywords])
            print(f"關鍵詞: {keywords_str}")

            # Related documents
            if texts and topics_list:
                topic_docs = [(i, doc) for i, (doc, doc_topic) in enumerate(zip(texts, topics_list)) if doc_topic == topic_id]
                if len(topic_docs) >= 2:
                    for doc_num, (doc_idx, doc_text) in enumerate(topic_docs[:2], 1):
                        display_text = doc_text[:80] + "..." if len(doc_text) > 80 else doc_text
                        print(f"  文檔{doc_num}: {display_text}")

            print("-" * 50)

    def save_manual_naming_results(self, dataset_name, enhanced_topics):
        """Safely save manual naming results"""
        if not enhanced_topics:
            print("錯誤: 沒有命名結果可保存")
            return None

        print(f"\n正在保存 {dataset_name} 的命名結果...")

        try:
            # 1. Save to global variables (for current session use)
            # Use consistent naming convention, ensuring overwrite of any existing naming
            globals()[f'{dataset_name}_enhanced'] = enhanced_topics
            print(f"完成: 已保存到記憶體變數: {dataset_name}_enhanced")

            # 2. Save to JSON file (permanent storage)
            json_path = os.path.join(self.results_dir, f'{dataset_name}_manual_enhanced_topics.json')

            # Prepare serializable data
            serializable_data = {}
            for topic_id, topic_data in enhanced_topics.items():
                serializable_data[str(topic_id)] = topic_data

            with open(json_path, 'w', encoding='utf-8') as f:
                json.dump(serializable_data, f, ensure_ascii=False, indent=2)
            print(f"完成: 已保存到檔案: {json_path}")

            # 3. Save naming history
            if dataset_name in self.manual_naming_history:
                history_path = os.path.join(self.results_dir, f'{dataset_name}_naming_history.json')
                with open(history_path, 'w', encoding='utf-8') as f:
                    json.dump(self.manual_naming_history[dataset_name], f, ensure_ascii=False, indent=2)
                print(f"完成: 已保存命名歷史: {history_path}")

            # 4. Create backup
            import shutil
            backup_path = json_path.replace('.json', f'_backup_{pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")}.json')
            shutil.copy2(json_path, backup_path)
            print(f"完成: 已創建備份檔案: {backup_path}")

            print(f"\n保存完成: 共保存了 {len(enhanced_topics)} 個主題的命名")
            print("=" * 60)
            print("保存位置:")
            print(f"   記憶體變數: {dataset_name}_enhanced")
            print(f"   主檔案: {os.path.basename(json_path)}")
            print(f"   備份檔案: {os.path.basename(backup_path)}")
            if dataset_name in self.manual_naming_history:
                print(f"   歷史記錄: {dataset_name}_naming_history.json")

            return json_path

        except Exception as e:
            print(f"錯誤: 保存失敗: {str(e)}")
            return None

    def safe_exit_instructions(self):
        """Display instructions for safe exit and saving"""
        print(f"\n安全退出與保存說明")
        print("=" * 60)
        print("如何安全結束並保存命名結果:")
        print("")
        print("1. 在任何命名流程中:")
        print("   - 輸入 'done' - 自動保存並退出")
        print("   - 使用 Ctrl+C - 會提示是否保存")
        print("")
        print("2. 手動保存:")
        print("   - 選擇選項 '5. 保存當前命名結果'")
        print("   - 系統會自動創建多重備份")
        print("")
        print("3. 保存位置:")
        print("   - 主檔案: results/{dataset_name}_manual_enhanced_topics.json")
        print("   - 備份檔案: results/{dataset_name}_manual_enhanced_topics_backup_*.json")
        print("   - 歷史記錄: results/{dataset_name}_naming_history.json")
        print("")
        print("4. 重新載入:")
        print("   - 下次使用時選擇 '4. 載入之前的命名結果'")
        print("   - 或使用 manual_namer.load_manual_naming_results('dataset_name')")
        print("")
        print("重要提醒:")
        print("   - 命名結果會自動保存到雲端硬碟")
        print("   - 重啟 Colab 後需要重新載入模型")
        print("   - 但命名結果會永久保存在檔案中")
        print("=" * 60)

    def load_manual_naming_results(self, dataset_name):
        """Load previously saved manual naming results"""
        json_path = os.path.join(self.results_dir, f'{dataset_name}_manual_enhanced_topics.json')

        if not os.path.exists(json_path):
            print(f"錯誤: 找不到 {dataset_name} 的人工命名結果")
            return None

        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                data = json.load(f)

            # Convert keys back to integers
            enhanced_topics = {int(k): v for k, v in data.items()}

            # Load to global variables
            globals()[f'{dataset_name}_enhanced'] = enhanced_topics

            print(f"完成: 已載入 {dataset_name} 的人工命名結果，共 {len(enhanced_topics)} 個主題")
            return enhanced_topics

        except Exception as e:
            print(f"錯誤: 載入失敗: {str(e)}")
            return None

    def compare_naming_results(self, model, dataset_name, llm_enhanced=None, manual_enhanced=None):
        """Compare LLM automatic naming and manual naming results"""
        print(f"\n命名結果比較 - {dataset_name}")
        print("=" * 80)

        topic_info = model.get_topic_info()
        non_outlier_topics = topic_info[topic_info['Topic'] != -1].sort_values("Count", ascending=False)

        # If enhanced_topics is not provided, try to get it automatically
        if llm_enhanced is None:
            llm_var = f'{dataset_name}_enhanced_llm'
            if llm_var in globals():
                llm_enhanced = globals()[llm_var]

        if manual_enhanced is None:
            manual_var = f'{dataset_name}_enhanced'
            if manual_var in globals():
                manual_enhanced = globals()[manual_var]

        comparison_data = []

        for _, row in non_outlier_topics.head(15).iterrows():
            topic_id = row['Topic']
            topic_size = row['Count']

            # Get keywords
            keywords = [word for word, _ in model.get_topic(topic_id)[:3]]
            keywords_str = ", ".join(keywords)

            # LLM Naming
            llm_name = "未命名"
            if llm_enhanced and topic_id in llm_enhanced:
                llm_name = llm_enhanced[topic_id]['name']

            # Manual Naming
            manual_name = "未命名"
            if manual_enhanced and topic_id in manual_enhanced:
                manual_name = manual_enhanced[topic_id]['name']

            # Mark if modified
            modified = "是" if manual_name != "未命名" and manual_name != llm_name else ""

            comparison_data.append([
                topic_id,
                topic_size,
                llm_name,
                manual_name,
                modified,
                keywords_str
            ])

        headers = ['主題ID', '文檔數', 'LLM命名', '人工命名', '已修改', '關鍵詞']
        print(tabulate(comparison_data, headers=headers, tablefmt="grid",
                      maxcolwidths=[None, None, 25, 25, None, 30]))

def manual_rename_workflow(dataset_name):
    """Manual rename workflow - Simplified version"""
    model_var = f'{dataset_name}_model'

    if model_var not in globals():
        print(f"錯誤: 模型 {model_var} 不存在")
        print("請先執行模型載入區塊")
        return None

    model = globals()[model_var]

    # Get display name
    if 'PIPELINE_CONFIG' in globals() and dataset_name in PIPELINE_CONFIG['data']['datasets']:
        display_name = PIPELINE_CONFIG['data']['datasets'][dataset_name]['description']
    else:
        display_name = dataset_name

    print(f"\n{display_name} 人工命名流程")
    print("=" * 60)

    # Initialize manual_namer (if not exists)
    if 'manual_namer' not in globals():
        if 'PIPELINE_CONFIG' in globals():
            globals()['manual_namer'] = ManualTopicNamer(PIPELINE_CONFIG)
        else:
            print("錯誤: PIPELINE_CONFIG 未定義，無法初始化 manual_namer")
            return None

    manual_namer = globals()['manual_namer']

    # Get current enhanced topics
    enhanced_topics, source = manual_namer.get_current_enhanced_topics(dataset_name)
    if enhanced_topics:
        print(f"當前命名狀態: {source} ({len(enhanced_topics)}個主題)")
    else:
        print(f"當前命名狀態: 無命名")

    # Display safe exit instructions
    manual_namer.safe_exit_instructions()

    # Select operation mode
    print(f"\n請選擇操作模式:")
    print("1. 查看前5個主題詳情")
    print("2. 批次命名模式 (推薦)")
    print("3. 單個主題重命名")
    print("4. 載入之前的命名結果")
    print("5. 保存當前命名結果")
    print("6. 比較LLM vs 人工命名")
    print("0. 安全退出")

    while True:
        try:
            choice = input(f"\n請選擇操作 (0-6): ").strip()

            if choice == '0':
                # Safe exit
                current_topics, _ = manual_namer.get_current_enhanced_topics(dataset_name)
                if current_topics:
                    save_choice = input("是否保存當前命名結果？(y/n): ").strip().lower()
                    if save_choice in ['y', 'yes', '是']:
                        manual_namer.save_manual_naming_results(dataset_name, current_topics)
                print("已安全退出")
                break

            elif choice == '1':
                manual_namer.display_current_topics(model, display_name)

            elif choice == '2':
                print(f"\n進入批次命名模式...")
                enhanced_topics = manual_namer.batch_rename_interface(model, dataset_name)

            elif choice == '3':
                manual_namer.display_current_topics(model, display_name)
                try:
                    topic_id = int(input(f"\n請輸入要重命名的主題ID: "))
                    new_name = input("請輸入新名稱: ").strip()

                    if new_name:
                        enhanced_topics = manual_namer.manual_rename_single(
                            model, dataset_name, topic_id, new_name
                        )
                        if enhanced_topics:
                            # Save to global variable immediately
                            globals()[f'{dataset_name}_enhanced'] = enhanced_topics
                except ValueError:
                    print("錯誤: 請輸入有效的主題ID")

            elif choice == '4':
                loaded_topics = manual_namer.load_manual_naming_results(dataset_name)
                if loaded_topics:
                    enhanced_topics = loaded_topics

            elif choice == '5':
                current_topics, _ = manual_namer.get_current_enhanced_topics(dataset_name)
                if current_topics:
                    manual_namer.save_manual_naming_results(dataset_name, current_topics)
                else:
                    print("錯誤: 沒有命名結果可保存")

            elif choice == '6':
                manual_namer.compare_naming_results(model, display_name)

            else:
                print("錯誤: 無效選擇，請輸入 0-6")

        except ValueError:
            print("錯誤: 請輸入有效數字")
        except KeyboardInterrupt:
            print(f"\n檢測到中斷信號")
            current_topics, _ = manual_namer.get_current_enhanced_topics(dataset_name)
            if current_topics:
                save_choice = input("是否保存當前命名結果？(y/n): ").strip().lower()
                if save_choice in ['y', 'yes', '是']:
                    manual_namer.save_manual_naming_results(dataset_name, current_topics)
            print("已安全退出")
            break
        except Exception as e:
            print(f"錯誤: 發生錯誤: {str(e)}")

    # Return final enhanced topics
    final_topics, _ = manual_namer.get_current_enhanced_topics(dataset_name)
    return final_topics

def regenerate_visualizations_with_manual_naming():
    """Regenerate visualizations using manual naming"""
    print("使用人工命名重新生成視覺化圖表...")

    if 'PIPELINE_CONFIG' not in globals():
        print("錯誤: PIPELINE_CONFIG 未定義")
        return

    results_dir = os.path.join(PIPELINE_CONFIG['data']['base_path'], PIPELINE_CONFIG['output']['results_dir'])
    manual_dir = os.path.join(results_dir, 'manual_naming')
    os.makedirs(manual_dir, exist_ok=True)

    # Get manual naming results
    manual_enhanced = {}
    for dataset in ['operation_positive', 'operation_negative', 'ip_positive', 'ip_negative', 'balance_update']:
        enhanced_var = f'{dataset}_enhanced'
        if enhanced_var in globals() and globals()[enhanced_var]:
            manual_enhanced[dataset] = globals()[enhanced_var]

    if not manual_enhanced:
        print("未找到任何人工命名結果")
        return

    # Regenerate individual topic distribution plots
    for dataset in manual_enhanced.keys():
        model_var = f'{dataset}_model'
        if model_var in globals():
            model = globals()[model_var]
            enhanced = manual_enhanced[dataset]
            save_path = os.path.join(manual_dir, f'{dataset}_manual_named.png')

            print(f"   生成 {dataset} 人工命名主題分布圖...")
            if 'visualizer' in globals():
                globals()['visualizer'].plot_topic_volume_semantic(
                    model,
                    PIPELINE_CONFIG['data']['datasets'][dataset]['description'] + " (人工命名)",
                    enhanced,
                    save_path
                )

    # Regenerate comparison plots
    if ('operation_positive' in manual_enhanced and 'operation_negative' in manual_enhanced and
        'operation_positive_model' in globals() and 'operation_negative_model' in globals()):
        save_path = os.path.join(manual_dir, 'operation_comparison_manual_named.png')
        print("   生成營運活動正負面比較圖 (人工命名)...")
        if 'visualizer' in globals():
            globals()['visualizer'].plot_operation_activity_comparison(
                globals()['operation_positive_model'],
                globals()['operation_negative_model'],
                manual_enhanced['operation_positive'],
                manual_enhanced['operation_negative'],
                save_path
            )

    if ('ip_positive' in manual_enhanced and 'ip_negative' in manual_enhanced and
        'ip_positive_model' in globals() and 'ip_negative_model' in globals()):
        save_path = os.path.join(manual_dir, 'ip_comparison_manual_named.png')
        print("   生成IP聯動正負面比較圖 (人工命名)...")
        if 'visualizer' in globals():
            globals()['visualizer'].plot_ip_collaboration_comparison(
                globals()['ip_positive_model'],
                globals()['ip_negative_model'],
                manual_enhanced['ip_positive'],
                manual_enhanced['ip_negative'],
                save_path
            )

    print(f"人工命名視覺化完成，圖表已儲存至: {manual_dir}")

# Shortcut functions
def quick_review_all():
    """Quickly view all model topics"""
    datasets = ['operation_positive', 'operation_negative', 'ip_positive', 'ip_negative', 'balance_update']

    for dataset in datasets:
        model_var = f'{dataset}_model'
        if model_var in globals():
            model = globals()[model_var]
            if 'PIPELINE_CONFIG' in globals():
                display_name = PIPELINE_CONFIG['data']['datasets'][dataset]['description']
            else:
                display_name = dataset

            if 'manual_namer' not in globals():
                if 'PIPELINE_CONFIG' in globals():
                    globals()['manual_namer'] = ManualTopicNamer(PIPELINE_CONFIG)
                else:
                    print("錯誤: PIPELINE_CONFIG 未定義")
                    return

            manual_namer = globals()['manual_namer']
            print(f"\n查看 {display_name}:")
            manual_namer.display_current_topics(model, display_name)

def quick_rename(dataset_name):
    """Quickly rename specified dataset"""
    return manual_rename_workflow(dataset_name)

# Initialize manual namer
if 'PIPELINE_CONFIG' in globals():
    manual_namer = ManualTopicNamer(PIPELINE_CONFIG)
    print("完成: 人工主題命名器已初始化")

    print("\n人工主題命名功能已就緒")
    print("=" * 50)
    print("可用功能:")
    print("quick_review_all() - 快速檢視所有模型主題")
    print("quick_rename('dataset_name') - 快速重命名指定資料集")
    print("manual_rename_workflow('dataset_name') - 完整重命名工作流程")
    print("regenerate_visualizations_with_manual_naming() - 重新生成視覺化圖表")
    print("\n可用資料集名稱:")
    print("'operation_positive' - 營運活動正面情緒")
    print("'operation_negative' - 營運活動負面情緒")
    print("'ip_positive' - IP聯動正面情緒")
    print("'ip_negative' - IP聯動負面情緒")
    print("'balance_update' - 平衡性版本更新")
    print("\n範例用法:")
    print("quick_review_all()  # 檢視所有主題")
    print("quick_rename('operation_positive')  # 重命名營運活動正面主題")
else:
    print("錯誤: PIPELINE_CONFIG 未定義，無法初始化人工主題命名器")


quick_rename('dataset_name')
manual_rename_workflow('dataset_name')
quick_rename('balance_update')

完成: 人工主題命名器已初始化

人工主題命名功能已就緒
可用功能:
quick_review_all() - 快速檢視所有模型主題
quick_rename('dataset_name') - 快速重命名指定資料集
manual_rename_workflow('dataset_name') - 完整重命名工作流程
regenerate_visualizations_with_manual_naming() - 重新生成視覺化圖表

可用資料集名稱:
'operation_positive' - 營運活動正面情緒
'operation_negative' - 營運活動負面情緒
'ip_positive' - IP聯動正面情緒
'ip_negative' - IP聯動負面情緒
'balance_update' - 平衡性版本更新

範例用法:
quick_review_all()  # 檢視所有主題
quick_rename('operation_positive')  # 重命名營運活動正面主題
錯誤: 模型 dataset_name_model 不存在
請先執行模型載入區塊
錯誤: 模型 dataset_name_model 不存在
請先執行模型載入區塊

平衡性版本更新 人工命名流程
當前命名狀態: 人工命名 (34個主題)

安全退出與保存說明
如何安全結束並保存命名結果:

1. 在任何命名流程中:
   - 輸入 'done' - 自動保存並退出
   - 使用 Ctrl+C - 會提示是否保存

2. 手動保存:
   - 選擇選項 '5. 保存當前命名結果'
   - 系統會自動創建多重備份

3. 保存位置:
   - 主檔案: results/{dataset_name}_manual_enhanced_topics.json
   - 備份檔案: results/{dataset_name}_manual_enhanced_topics_backup_*.json
   - 歷史記錄: results/{dataset_name}_naming_history.json

4. 重新載入:
   - 下次使用時選擇 '4. 載入之前的命名結果'
   - 或使用 manual_namer.load_manual_naming_results('

{0: {'name': '黑貓無敵主播討論',
  'keywords': ['辛苦 黑貓 無敵',
   '黑貓 無敵 主播',
   '貴 黑貓 無敵',
   '主播 黑貓 無敵',
   '辛苦 主播 黑貓',
   '貴 黑貓',
   '辛苦 黑貓',
   '無敵 黑貓'],
  'original_keywords': ['辛苦 黑貓 無敵(0.869)',
   '黑貓 無敵 主播(0.864)',
   '貴 黑貓 無敵(0.863)',
   '主播 黑貓 無敵(0.850)',
   '辛苦 主播 黑貓(0.847)']},
 1: {'name': '黃金戰神外觀討論',
  'keywords': ['黃金 戰神',
   '帥 黃金 戰神',
   '辛苦 黃金 戰神',
   '造型 黃金 戰神',
   '開箱 黃金 戰神',
   '休息 黃金 戰神',
   '黃金 戰神 遊戲',
   '主播 黃金 戰神'],
  'original_keywords': ['黃金 戰神(0.962)',
   '帥 黃金 戰神(0.946)',
   '辛苦 黃金 戰神(0.942)',
   '造型 黃金 戰神(0.925)',
   '開箱 黃金 戰神(0.922)']},
 2: {'name': '技能削弱討論',
  'keywords': ['削弱', '弱', '決定 削弱', '技能', '強化', '技', '加強', '技能組'],
  'original_keywords': ['削弱(0.764)',
   '弱(0.748)',
   '決定 削弱(0.708)',
   '技能(0.665)',
   '強化(0.655)']},
 3: {'name': '遊戲平衡抱怨',
  'keywords': ['煩', '哭 哭', '哭', '嗆', '破', '慘', '蠻', '爛'],
  'original_keywords': ['煩(0.763)',
   '哭 哭(0.738)',
   '哭(0.717)',
   '嗆(0.699)',
   '破(0.687)']},
 4: {'name': '莉莉安選中與削弱討論',
  'keywords': ['莉莉安',
   '莉莉安 不可 選中'